# പാഠം 10 - ഉത്പാദനത്തിൽ AI ഏജന്റുകൾ

ഈ പಾಠത്തിൽ നിങ്ങൾ **ഉത്പാദന മാതൃകകൾ** പഠിക്കാനും Microsoft Agent Framework (Python) ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കുള്ളവ. നാം പരിഗണിക്കുന്നത്:

- **നിരീക്ഷണക്ഷമത** — ഏജന്റ് ഇടപെടലുകളിൽ സമയവും ലോഗിങ്ങും ചേർക്കൽ
- **അവലോകനം** — പ്രതികരണ ഗുണനിലവാരം സ്കോർ ചെയ്യാൻ ഒരു അവലോകന ഏജന്റ് ഉപയോഗിക്കൽ
- **വിലയിരുത്തൽ** — ടോക്കൺ മിതവിനിയോഗവും മോഡൽ തിരഞ്ഞെടുപ്പും സംബന്ധിച്ച തന്ത്രങ്ങൾ

സംഭവം ഒരു **പ്രവാസ ഏജന്റ്** ആണ്, ഇത് ഉപയോക്താക്കളെ യാത്രകൾ പ്ലാൻ ചെയ്യാൻ സഹായിക്കുകയും, മോണിറ്ററിങ്ങും അവലോകനവും മേൽനോട്ടത്തിലാക്കുകയും ചെയ്യുന്നു.


## സെറ്റപ്പ്


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ഉത്പാദന പരിഗണനകൾ

പ്രോടോടൈപ്പുകളിൽ നിന്നുള്ള AI ഏജന്റ്‌മാരെ ഉത്പാദനത്തിലേക്ക് മാറ്റുന്നത് കൃത്യമായ ശ്രദ്ധ ആവശ്യമാണ് മൂന്ന് പ്രധാന തൂണുകളിൽ:

1. **പര്യാസനീയത** — ഏജന്റ് എന്ത് ചെയ്യുന്നു, എത്രസമയം ചെലവഴിക്കുന്നു, ഏത് ഉപകരണങ്ങൾ വിളിക്കുന്നു என்பതിൽ നിനക്ക് ദൃശ്യമായ കാണൽ വേണം. ട്രേസിംഗ്, ലോ​ഗിംഗ് ഇല്ലാതെ ഉത്പാദനത്തിലെ പ്രശ്നങ്ങൾ നന്നായി വിലയിരുത്തി പരിഹരിക്കാനാകില്ല.

2. **വിലയിരുത്തൽ** — ഓട്ടോമേറ്റഡ് ഗുണനിലവാര പരിശോധനകൾ ഏജന്റെ പ്രതികരണങ്ങൾ സമയം പൂർണ്ണതയോടെ, ശരിയായി, ഉപകാരപ്രദമായി തുടരുന്നതു ഉറപ്പുവരുത്തുന്നു. ഒരു വിലയിരുത്തൽ ഏജന്റ് നിർവചിച്ച മാനദണ്ഡങ്ങളോടനുസരിച്ച് പ്രതികരണങ്ങളെ സ്കോർ ചെയ്യുന്നു.

3. **വ്യയം നിയന്ത്രണം** — ടോക്കൺ ഉപഭോഗം നേരിട്ട് ചെലവിൽ ബാധിക്കുന്നു. പ്രാമ്പ്റ്റ് മെച്ചപ്പെടുത്തി, മോഡൽ തിരഞ്ഞെടുക്കുകയും കാഷിംഗ് നടപ്പിലാക്കുകയും ചെയ്യുന്ന തന്ത്രങ്ങൾ ഗുണമേന്മ പ്രശ്നമില്ലാതെ ചെലവുകൾ നിയന്ത്രിക്കുന്നതിനായി സഹായിക്കുന്നു.


## ഒരു ഓബ്ജർവബിൾ ഏജന്റ് നിർമ്മിക്കൽ

നാം യാത്രാ ഉപകരണങ്ങൾ നിർവ്വചിച്ച് ലാറ്റൻസി നിരീക്ഷിക്കാൻ ഏജന്റ് کالിനൊപ്പം ടൈമിംഗ് ചേർത്ത് രാപ്തിയിലാണ്. പ്രൊഡക്ഷനിൽ, നിങ്ങൾ OpenTelemetry അല്ലെങ്കിൽ സമാനമായ ടീസിംഗ് ബാക്ക്എൻഡുമായി സംയോജിപ്പിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## വിലയിരുത്തൽ മാതൃകകൾ

ഒരു പൊതുവായ ഉൽപാദന മാതൃക എന്നത് രണ്ടാം ഏജന്റ് ഒരു **വിലയിരുത്തുന്ന വ്യക്തി** ആയി ഉപയോഗപ്പെടുത്തുന്നത് ആണ്. വിലയിരുത്തുന്ന ব্যক্তি പ്രധാന ഏജന്റിന്റെ പ്രതികരണത്തെ പൂര്‍ത്തിമാനത, ശുദ്ധത, സഹായകത തുടങ്ങിയ നിർവ്വചിച്ച മാനദണ്ഡങ്ങളെ അടിസ്ഥാനമാക്കി സ്കോർ ചെയ്യുന്നു.

ഇത് സാധ്യമാക്കുന്നത്:
- ഉപയോക്താക്കളിലേക്ക് പ്രതികരണങ്ങൾ എത്തുന്നതിനു മുന്നേ സ്വയം ചെയ്‌ത ഗുണനിലവാര പരിശോധനകൾ
- പ്രോംപ്റ്റുകൾ അല്ലെങ്കിൽ മാതൃകകൾ മാറുമ്പോൾ പണംപ്പാളി കണ്ടെത്തൽ
- ഏജന്റിന്റെ പ്രവർത്തനം സ്ഥിരമായി നിരീക്ഷിക്കൽ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ചെലവ് നിയന്ത്രണ തന്ത്രങ്ങൾ

ഉൽപ്പാദന AI ഏജന്റുകൾക്കായി ചെലവ് നിയന്ത്രിക്കുന്നത് അത്യന്താപേക്ഷിതമാണ്. പ്രധാന തന്ത്രങ്ങൾ ഇവയാണ്:

| തന്ത്രം | വിവരണം |
|---|---|
| **പ്രോംപ്റ്റ് മെച്ചപ്പെടുത്തൽ** | സിസ്റ്റം നിർദ്ദേശങ്ങൾ ലഘൂകരിച്ച് സൂക്ഷിക്കുക. ഇൻപുട്ട് ടോക്കണുകൾ കുറക്കാൻ ആവർത്തന ഉപയോഗം ഒഴിവാക്കുക. |
| **മോഡൽ തിരഞ്ഞെടുപ്പ്** | ക്ലാസിഫിക്കേഷൻ അല്ലെങ്കിൽ എൻട്രാക്ഷൻ പോലുള്ള ലഘുവായ ജോലികൾക്കായി ചെലവുകുറഞ്ഞ ചെറിയ മോഡലുകൾ (ഉദാ: GPT-4o-mini) ഉപയോഗിക്കുക, സങ്കീേർണ്ണമായ സൗചനങ്ങൾക്ക് വലിയ മോഡലുകൾ സൂക്ഷിക്കുക. |
| **കാഷിംഗ്** | ഉപകരണം ഫലങ്ങളും ആവർത്തിക്കുന്ന ചോദ്യങ്ങളും കാഷ് ചെയ്യുക, ആവർത്തന API കാൾ ഒഴിവാക്കാൻ. |
| **ടോക്കൺ ബഡ്ജറ്റുകൾ** | അത്ഭുതകരമായി ദൈർഘ്യമേറിയ മറുപടികൾ തಡೆಯാൻ `max_tokens` പരിധികൾ സ്ഥാപിക്കുക. |
| **ബാച്ചിംഗ്** | കഴിയുന്നിടത്ത് ചേർന്ന ഉപയോക്തൃ ചോദനകൾ ഒരു ഏക ഏക API കോളിൽ ഓർക്കുക. |

പ്രായോഗികമായി, പാളിയിട്ട സമീപനം ഫലപ്രദമാണ്: നേരിട്ടുള്ള അഭ്യർത്ഥനകൾ വേഗത്തിലുള്ള, കുറവ് ചെലവുള്ള മോഡലിലേക്ക് വഴിമാറിച്ച് സങ്കീർണ്ണമായ ചോദനകൾ മെച്ചപ്പെട്ട കഴിവുള്ളവയ്ക്ക് മാത്രം ഉയർത്തുക.


## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. ടൈമിംഗും ലോഗിംഗും ഉപയോഗിച്ച് ഏജന്റ് ഇടപെടലുകളിൽ **പരീക്ഷണക്ഷമത ചേർക്കുന്നത്**, ട്രേസിംഗിനും മോണിറ്ററിംഗിനും അടിസ്ഥാനമിട്ട ഒരു കാര്യം.
2. പൂർത്തിയായ തുല്യത, കൃത്യത, സഹായകരത എന്നിവയുടെ അളവ് നിർണ്ണയിക്കുന്ന ഒരു മൂല്യനിർണ്ണയ ഏജന്റിന്റെ സഹായത്തോടെ ഏജന്റ് പ്രതികരണങ്ങൾ **സ്വയം അളക്കുന്നത്**.
3. പ്രോംപ്റ്റ് ഓപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, കാഷിംഗ്, ടോക്കൺ ബജറ്റുകൾ എന്നിവ മുഖേന **ചെലവുകൾ നിയന്ത്രിക്കുന്നത്**.

ഈ നിർമ്മാണ മാതൃകകൾ നിങ്ങളുടെ AI ഏജന്റുകൾ വിശ്വസനീയവും അളക്കാനുമായതും വിപുലമായി ചെലവുലാഭകരവുമാക്കാൻ സഹായിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
